# 15 Karnataka spatial ML: spatial lags, spatial cross-validation and residual maps

Question (plan item 7, task T282): does knowing the neighbours help the M21 rain-anomaly model,
and are our error bars honest when districts move together?

- Spatial lags: mean over a district's queen neighbours of the anomaly, SPI-1 and SPI-3 of month
  t-1. A neighbour's value for month t is never a feature (it is not known at issue time).
- M21 LightGBM (T281 procedure) with and without the lags, fitted on train 1982-2014 and scored
  once on tune 2015-2017. Comparison only: the named candidate (T283) is unchanged and the test
  (2020-2025) stays closed.
- Cross-validation inside train (2005-01 to 2014-11): random 5-fold, leave one region out,
  yearly rolling origins, and both together, to show how much leaky schemes overstate skill.
- Moran's I of tune residuals per month before and after the lags (999 permutations, BH).
- M05 spatial challenger: v1 ridge plus neighbour SPI at t-1 on the D97 rolling origins, months
  before 2020 only (2020-2024 was the M05 test), research-only.

Inputs: the M21 dataset (hash-checked against `models/gates/m21.json`), the M05 dataset rebuilt
from local S08 IMD and S17 raw files (hash-checked against the saved v1 fit), the T109 queen
neighbours and the committed district outlines (centroids for the regions). Only derived indices
leave this notebook; no IMD mm value is written. Hindcast (D39).

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/15_ka_spatial_ml/` (or `ATLAS_EDA_OUT`).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import shapely.geometry
from matplotlib.colors import TwoSlopeNorm

from atlas.models import spatial_ml

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/15_ka_spatial_ml"))
result = spatial_ml.build(OUT)
s = result["summary"]
print(json.dumps({k: s["tune"][k] for k in ("lags_over_base", "lag_helps")}, indent=1))
print(json.dumps({k: s["moran"][k] for k in ("before", "after", "effective_size")}, indent=1))

In [ ]:
by = [r for r in s["moran"]["by_month"] if "before_i" in r]
fig, ax = plt.subplots(figsize=(8, 4))
xs = np.arange(len(by))
ax.axhline(-1 / 29, color="grey", lw=0.8, label="E[I] = -1/29")
ax.plot(xs - 0.12, [r["before_i"] for r in by], "o", ms=4, label="LightGBM, T281 features")
ax.plot(xs + 0.12, [r["after_i"] for r in by], "s", ms=4, label="LightGBM + spatial lags")
ax.set_xticks(xs[::3], [r["month"] for r in by[::3]], rotation=45, fontsize=7)
ax.set_ylabel("Moran's I of tune residuals")
ax.set_title("M21 tune residuals stay clustered with or without spatial lags", fontsize=10)
ax.legend(fontsize=7)
fig.text(
    0.01,
    0.01,
    "S08 IMD hindcast, 30 S17 districts, queen weights; months with every district scored "
    f"({len(by)} of 36), 999 permutations. Anomaly units.",
    fontsize=6,
)
fig.subplots_adjust(bottom=0.25)
fig.savefig(OUT / "moran_tune.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
schemes = s["cv"]["schemes"]
fig, ax = plt.subplots(figsize=(9, 4))
ys = np.arange(len(schemes))
for off, key, label in ((-0.15, "base", "T281 features"), (0.15, "lags", "+ spatial lags")):
    ci = [v[key]["improvement_over_climatology"] for v in schemes.values()]
    mean = np.array([c["mean"] for c in ci])
    err = np.array([[c["mean"] - c["lower95"] for c in ci], [c["upper95"] - c["mean"] for c in ci]])
    ax.errorbar(mean, ys + off, xerr=err, fmt="o", capsize=3, label=label)
t = s["tune"]["models"]["base"]["improvement_over_climatology"]["mean"]
ax.axvline(0, color="grey", lw=0.8)
ax.axvline(t, color="black", ls=":", lw=1, label="LightGBM on tune 2015-2017")
ax.set_yticks(ys, [v["label"] for v in schemes.values()], fontsize=8)
ax.set_xlabel("MAE improvement over climatology (anomaly units, 95% interval)")
ax.set_title("Cross-validation inside train: what each scheme claims", fontsize=10)
ax.legend(fontsize=7)
fig.text(0.01, 0.01, f"M21 LightGBM, {s['cv']['scored']}; blocks = months.", fontsize=6)
fig.subplots_adjust(left=0.4, bottom=0.18)
fig.savefig(OUT / "cv_schemes.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
shapes = {
    f["properties"]["place_id"]: shapely.geometry.shape(f["geometry"])
    for f in json.loads(spatial_ml.OUTLINES.read_text("utf-8"))["features"]
}


def draw(ax, geom, color):
    for poly in getattr(geom, "geoms", [geom]):
        x, y = poly.exterior.xy
        ax.fill(x, y, color=color, ec="white", lw=0.5)


gain = {d["place_id"]: d["improvement_base"] for d in s["districts"]}
lim = max(abs(v) for v in gain.values())
norm, cmap = TwoSlopeNorm(0, -lim, lim), plt.get_cmap("RdBu")
names = sorted(set(result["region_of"].values()))
colors = dict(zip(names, plt.get_cmap("tab10").colors, strict=False))
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 5.5))
for p, geom in shapes.items():
    draw(a, geom, colors[result["region_of"][p]])
    draw(b, geom, cmap(norm(gain[p])))
for r in names:
    a.plot([], [], "s", color=colors[r], label=r)
a.legend(fontsize=7, loc="lower left")
a.set_title("Cross-validation regions", fontsize=9)
b.set_title("Tune MAE improvement over climatology", fontsize=9)
fig.colorbar(plt.cm.ScalarMappable(norm, cmap), ax=b, shrink=0.6, label="anomaly units")
for ax in (a, b):
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.text(
    0.01,
    0.01,
    "M21 LightGBM (T281 features), tune 2015-2017, S08 IMD hindcast; blue = better than "
    "climatology. Outlines: S17 geoBoundaries 2021 (ODbL), simplified.",
    fontsize=6,
)
fig.savefig(OUT / "district_map.png", dpi=80, metadata={"Software": None})
plt.close(fig)
c = s["m05_challenger"]
print(json.dumps({k: c[k] for k in ("mae", "spatial_over_ridge", "status")}, indent=1))

## Reading and limits

See `summary.json` (`tune`, `moran`, `cv`, `m05_challenger`, `limits`) and the T282 section of
`reports/models/m21-tune.md`. The residual clustering is mostly weather shared in the same
month, which no lagged feature can know; the honest output is the cross-validation that holds
out whole regions and moves forward in time, and intervals that resample whole months. Not an
emergency service.